# BhuNirvighna Land Acquisition Dataset
## EDA and Preprocessing

This notebook explores the raw land acquisition dataset and prepares the data for further analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### 1. Load the raw dataset

In [ ]:
file_path = 'data/raw/BhuNirvighna_Land_Acquisition_Dataset.csv'
df = pd.read_csv(file_path)

df.head()

### 2. Basic dataset information

In [ ]:
print('Shape of dataset:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())

df.info()

### 3. Check missing values and duplicates

In [ ]:
print('Missing values by column:')
print(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())

### 4. Numerical summary

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols].describe().T

### 5. Categorical columns

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    print(f'\n{col}:')
    print(df[col].value_counts().head(10))

### 6. Project type distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='Project_Type', order=df['Project_Type'].value_counts().index)
plt.xticks(rotation=45)
plt.title('Projects by Project Type')
plt.xlabel('Project Type')
plt.ylabel('Number of Projects')
plt.tight_layout()
plt.show()

### 7. Projects by state

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, y='State', order=df['State'].value_counts().index)
plt.title('Projects by State')
plt.xlabel('Number of Projects')
plt.ylabel('State')
plt.tight_layout()
plt.show()

### 8. Convert date columns

In [ ]:
date_cols = [
    'Sanction_Date', '3A_Notification_Date', '3D_Notification_Date',
    'Award_Date', 'Payment_Date', 'Possession_Date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df[date_cols].dtypes

### 9. Feature creation

In [ ]:
df['Land_Acquisition_Ratio'] = (
    df['Land_Acquired_Hectare'] / df['Land_Required_Hectare']
)

df['Displacement_Ratio'] = (
    df['Displaced_Families'] / df['Affected_Families']
)

df['Total_Compensation_Crore'] = (
    df['Compensation_Amount_Crore'] + df['RR_Amount_Crore']
)

df[['Land_Acquisition_Ratio', 'Displacement_Ratio', 'Total_Compensation_Crore']].head()

### 10. Prepare the preprocessed dataset

In [ ]:
processed_df = df.copy()

# Project_ID is an identifier, while Data_Basis is descriptive metadata.
processed_df = processed_df.drop(columns=['Project_ID', 'Data_Basis'])

# Extract useful information from dates.
for col in date_cols:
    processed_df[col + '_Year'] = processed_df[col].dt.year
    processed_df[col + '_Month'] = processed_df[col].dt.month

processed_df = processed_df.drop(columns=date_cols)

# Encode categorical variables.
processed_df = pd.get_dummies(processed_df, drop_first=True)

# Fill missing numeric values using the median.
for col in processed_df.select_dtypes(include=np.number).columns:
    processed_df[col] = processed_df[col].fillna(processed_df[col].median())

processed_df.head()

### 11. Final preprocessing check

In [ ]:
print('Processed dataset shape:', processed_df.shape)
print('Total missing values:', processed_df.isnull().sum().sum())

processed_df.describe().T